In [1]:
import copy
import re
import tempfile
import ase.io.cube
import ipywidgets as ipw
import nglview as nv
import numpy as np
import traitlets as tl
from aiida import orm, plugins
from cubehandler import Cube
from PIL import ImageColor
from ase.io import read
from ase import io
import nglview as nv
import ipywidgets as widgets

In [37]:
import os
from ipywidgets import Layout

cube_folder = sorted([name for name in os.listdir("./cubes") if name.endswith(".cube")])

cube_file_1 = widgets.Dropdown(
    options=cube_folder,
    value=cube_folder[0] if cube_folder else None,
    description="Molecule 1"
 )

cube_file_2 = widgets.Dropdown(
    options=cube_folder,
    value=cube_folder[1] if len(cube_folder) > 1 else (cube_folder[0] if cube_folder else None),
    description="Molecule 2"
 )

# Initialize views with current dropdown selections
CUBE_PATH_1 = f"./cubes/{cube_file_1.value}"
CUBE_PATH_2 = f"./cubes/{cube_file_2.value}"
acrolein14 = ase.io.read(CUBE_PATH_1)
acrolein15 = ase.io.read(CUBE_PATH_2)

molecule1 = nv.show_ase(acrolein14)
molecule2 = nv.show_ase(acrolein15)
NTO_1 = molecule1.add_component(CUBE_PATH_1, ext="cube")
NTO_2 = molecule2.add_component(CUBE_PATH_2, ext="cube")

molecule1.layout = widgets.Layout(width="100%", flex="1 1 50%", min_width="0")
molecule2.layout = widgets.Layout(width="100%", flex="1 1 50%", min_width="0")

positive_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="+ isovalue",
    readout_format=".1e",
 )
negative_level = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-5,
    max=-1,
    step=0.1,
    description="- isovalue",
    readout_format=".1e",
 )
positive_color = widgets.ColorPicker(value="#3366ff", description="+ color")
negative_color = widgets.ColorPicker(value="#ff4d4d", description="- color")
status = widgets.HTML()

def redraw_NTO_1(_=None):
    NTO_1.clear()
    NTO_1.add_surface(
        color=positive_color.value,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_1.add_surface(
        color=negative_color.value,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_NTO_2(_=None):
    NTO_2.clear()
    NTO_2.add_surface(
        color=positive_color.value,
        isolevelType="value",
        isolevel=float(positive_level.value),
        opacity=0.5,
    )
    NTO_2.add_surface(
        color=negative_color.value,
        isolevelType="value",
        isolevel=-float(negative_level.value),
        opacity=0.5,
    )
def redraw_surfaces(_=None):
    redraw_NTO_1()
    redraw_NTO_2()

def update_NTO_1(_=None):
    global NTO_1
    NTO_1.clear()
    NTO_1 = molecule1.add_component(cube_file_1.value, ext="cube")
    redraw_NTO_1()
    
def update_NTO_2(_=None):
    global NTO_2
    NTO_2.clear()
    NTO_2 = molecule2.add_component(cube_file_2.value, ext="cube")
    redraw_NTO_2()

for control in [positive_level, negative_level, positive_color, negative_color]:
    control.observe(redraw_surfaces, names="value")

cube_file_1.observe(update_NTO_1, names="value")
cube_file_2.observe(update_NTO_2, names="value")

redraw_surfaces()

controls = widgets.VBox(
    [
        widgets.HTML("<b>Cube isosurfaces</b>"),
        widgets.VBox([positive_color, negative_color], layout = Layout(border="1px solid black", width="40%")),
        status,
    ],
 )
molecule1_box = widgets.VBox([cube_file_1, molecule1], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule2_box = widgets.VBox([cube_file_2, molecule2], layout= Layout(width="100%", flex="1 1 50%", min_width="0"))
molecule_box = widgets.HBox([molecule1_box, molecule2_box], layout=Layout(width="100%", flex="1 1 100%"))

widgets.VBox([controls, molecule_box], layout = Layout(width="100%", flex="1 1 100%"))

In [36]:
print(molecule1.layout)

Layout(height='500px', min_width='0', width='100%')


In [3]:
molecule1.center_view?

Signature: molecule1.center_view(*args, **kwargs)
Docstring: <no docstring>
File:      ~/miniconda3/envs/atmospec/lib/python3.12/site-packages/nglview/widget.py
Type:      method

In [4]:
controls.observe?

Signature:
controls.observe(
    handler: 't.Callable[..., t.Any]',
    names: 'Sentinel | str | t.Iterable[Sentinel | str]' = traitlets.All,
    type: 'Sentinel | str' = 'change',
) -> 'None'
Docstring:
Setup a handler to be called when a trait changes.

This is used to setup dynamic notifications of trait changes.

Parameters
----------
handler : callable
    A callable that is called when a trait changes. Its
    signature should be ``handler(change)``, where ``change`` is a
    dictionary. The change dictionary at least holds a 'type' key.
    * ``type``: the type of notification.
    Other keys may be passed depending on the value of 'type'. In the
    case where type is 'change', we also have the following keys:
    * ``owner`` : the HasTraits instance
    * ``old`` : the old value of the modified trait attribute
    * ``new`` : the new value of the modified trait attribute
    * ``name`` : the name of the modified trait attribute.
names : list, str, All
    If names is All, the 